# **07 - Test Model with External Data**
Notebook ini untuk testing model dengan data eksternal (data yang belum pernah dilihat model saat training)

## **Import Library**

In [ ]:
import os
import glob
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from torchvision import transforms, models
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
    auc
)
from collections import Counter
import json
import time
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Path
EXTERNAL_DATA_DIR = "../data/external"  # Folder untuk external test data
SAVED_MODELS_DIR = "../saved_models"
RESULTS_DIR = "../results/external_test"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(EXTERNAL_DATA_DIR, exist_ok=True)

# Class mapping
ORGANIC_CLASSES = ['Food Organics', 'Vegetation']
INORGANIC_CLASSES = ['Cardboard', 'Glass', 'Metal', 'Miscellaneous Trash', 'Paper', 'Plastic', 'Textile Trash']
ALL_CLASSES = ORGANIC_CLASSES + INORGANIC_CLASSES

print("✓ Library berhasil dimuat!")

## **1. Setup: Struktur Folder External Data**

In [ ]:
print("="*70)
print("PERSIAPAN DATA EKSTERNAL")
print("="*70)

print("""
╔════════════════════════════════════════════════════════════════════╗
║  STRUKTUR FOLDER UNTUK EXTERNAL DATA:                              ║
╠════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  data/external/                                                    ║
║  ├── test_images/                  [Folder utama test images]     ║
║  │   ├── image1.jpg                                               ║
║  │   ├── image2.png                                               ║
║  │   └── ...                                                      ║
║  │                                                                ║
║  └── test_with_labels/             [OPTIONAL: Jika ada ground truth]║
║      ├── Food Organics/                                           ║
║      │   ├── img1.jpg                                             ║
║      │   └── img2.jpg                                             ║
║      ├── Vegetation/                                              ║
║      │   └── img3.jpg                                             ║
║      ├── Plastic/                                                 ║
║      │   └── img4.jpg                                             ║
║      └── ...                                                      ║
║                                                                    ║
╚════════════════════════════════════════════════════════════════════╝
""")

print("\n📂 Folder yang tersedia sekarang:")
if os.path.exists(EXTERNAL_DATA_DIR):
    subdirs = os.listdir(EXTERNAL_DATA_DIR)
    if subdirs:
        for subdir in subdirs:
            path = os.path.join(EXTERNAL_DATA_DIR, subdir)
            if os.path.isdir(path):
                files = glob.glob(os.path.join(path, '**', '*'), recursive=True)
                file_count = len([f for f in files if os.path.isfile(f)])
                print(f"   ├── {subdir}/ ({file_count} files)")
    else:
        print("   (folder kosong - silakan tambahkan data eksternal)")
else:
    print("   (folder belum dibuat)")

print("\n💡 Instruksi:")
print("   1. Buat folder 'data/external/test_images/' untuk images tanpa label")
print("   2. ATAU buat 'data/external/test_with_labels/' dengan struktur per-class")
print("   3. Notebook ini akan auto-detect struktur")

## **2. Load Pre-trained Models**

In [ ]:
print("="*70)
print("MEMUAT MODEL")
print("="*70)

# Check if model files exist
model_files = {
    'root': os.path.join(SAVED_MODELS_DIR, 'root_model_best.pth'),
    'organic': os.path.join(SAVED_MODELS_DIR, 'sub_organic_best.pth'),
    'inorganic': os.path.join(SAVED_MODELS_DIR, 'sub_inorganic_best.pth')
}

missing_models = []
for model_name, model_path in model_files.items():
    if os.path.exists(model_path):
        print(f"✓ {model_name}: {model_path}")
    else:
        print(f"✗ {model_name}: NOT FOUND")
        missing_models.append(model_name)

if missing_models:
    print(f"\n⚠️  Warning: Model {missing_models} belum dilatih!")
    print("   Jalankan notebook 03, 04 terlebih dahulu.")

print("\n" + "="*70)
print("MEMBANGUN ARSITEKTUR MODEL")
print("="*70)

# Image transforms (SAMA dengan training)
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def build_root_model():
    """Build ViT model untuk root classification"""
    print("\n→ Building Root Model (Vision Transformer)...")
    model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
    num_classes = 2  # organic, inorganic
    model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    return model

def build_organic_model():
    """Build MobileNetV3 model untuk organic classification"""
    print("\n→ Building Organic Model (MobileNetV3)...")
    model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
    num_classes = len(ORGANIC_CLASSES)
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
    return model

def build_inorganic_model():
    """Build ConvNeXt model untuk inorganic classification"""
    print("\n→ Building Inorganic Model (ConvNeXt-Tiny)...")
    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    num_classes = len(INORGANIC_CLASSES)
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
    return model

# Build models
print()
model_root = build_root_model()
model_organic = build_organic_model()
model_inorganic = build_inorganic_model()

# Load weights jika tersedia
models_to_load = [
    (model_root, model_files['root'], 'root'),
    (model_organic, model_files['organic'], 'organic'),
    (model_inorganic, model_files['inorganic'], 'inorganic')
]

for model, path, name in models_to_load:
    if os.path.exists(path):
        try:
            model.load_state_dict(torch.load(path, map_location=device))
            print(f"✓ Loaded {name} weights dari {path}")
        except Exception as e:
            print(f"✗ Error loading {name}: {e}")
    else:
        print(f"⚠️  {name} weights tidak ditemukan, menggunakan pre-trained only")

# Move ke device dan set eval mode
model_root.to(device).eval()
model_organic.to(device).eval()
model_inorganic.to(device).eval()

print("\n✓ Semua model berhasil dimuat dan di-set ke eval mode")

## **3. Load External Test Data**

In [ ]:
print("="*70)
print("MEMUAT DATA EKSTERNAL")
print("="*70)

def load_external_data_with_labels():
    """
    Load data dengan struktur per-class folder
    """
    print("\n→ Scanning data dengan labels (per-class folders)...")
    
    test_labels_dir = os.path.join(EXTERNAL_DATA_DIR, 'test_with_labels')
    
    if not os.path.exists(test_labels_dir):
        print(f"✗ Folder tidak ditemukan: {test_labels_dir}")
        return None
    
    image_paths = []
    true_labels = []
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG')
    
    class_dirs = [d for d in os.listdir(test_labels_dir) 
                  if os.path.isdir(os.path.join(test_labels_dir, d))]
    
    for class_name in sorted(class_dirs):
        class_path = os.path.join(test_labels_dir, class_name)
        for ext in extensions:
            for img_path in glob.glob(os.path.join(class_path, ext)):
                image_paths.append(img_path)
                true_labels.append(class_name)
    
    if len(image_paths) > 0:
        df = pd.DataFrame({
            'image_path': image_paths,
            'true_label': true_labels
        })
        return df
    return None

def load_external_data_without_labels():
    """
    Load data tanpa labels (semua images dalam satu folder)
    """
    print("\n→ Scanning data tanpa labels (test_images folder)...")
    
    test_dir = os.path.join(EXTERNAL_DATA_DIR, 'test_images')
    
    if not os.path.exists(test_dir):
        print(f"✗ Folder tidak ditemukan: {test_dir}")
        return None
    
    image_paths = []
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG')
    
    for ext in extensions:
        for img_path in glob.glob(os.path.join(test_dir, ext)):
            image_paths.append(img_path)
    
    if len(image_paths) > 0:
        df = pd.DataFrame({
            'image_path': image_paths,
            'true_label': [None] * len(image_paths)
        })
        return df
    return None

# Try load dengan labels dulu
df_external = load_external_data_with_labels()

if df_external is None:
    df_external = load_external_data_without_labels()

if df_external is not None:
    print(f"\n✓ Loaded {len(df_external)} images")
    print(f"\nSample data:")
    print(df_external.head())
    
    if df_external['true_label'].notna().any():
        print(f"\n✓ Labels tersedia!")
        print(f"\nDistribusi label:")
        print(df_external['true_label'].value_counts())
    else:
        print(f"\n⚠️  Labels tidak tersedia (untuk inference only)")
else:
    print("\n✗ Tidak ada data eksternal ditemukan!")
    print("   Silakan persiapkan folder sesuai instruksi di cell sebelumnya.")

## **4. Inference Function**

In [ ]:
def predict_hierarchical(image_path, model_root, model_org, model_inorg, data_transforms):
    """
    Predict menggunakan hierarchical system:
    1. Root model -> organic atau inorganic
    2. Sub model -> specific class
    """
    try:
        # Load dan preprocess image
        img = Image.open(image_path).convert('RGB')
        img_tensor = data_transforms(img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            # Step 1: Root prediction
            root_output = model_root(img_tensor)
            root_probs = torch.softmax(root_output, dim=1)
            root_pred_idx = torch.argmax(root_output, dim=1).item()
            root_pred = ['organic', 'inorganic'][root_pred_idx]
            root_conf = root_probs[0, root_pred_idx].item()
            
            # Step 2: Sub model prediction
            if root_pred == 'organic':
                sub_output = model_org(img_tensor)
                sub_classes = ORGANIC_CLASSES
            else:
                sub_output = model_inorg(img_tensor)
                sub_classes = INORGANIC_CLASSES
            
            sub_probs = torch.softmax(sub_output, dim=1)
            sub_pred_idx = torch.argmax(sub_output, dim=1).item()
            sub_pred = sub_classes[sub_pred_idx]
            sub_conf = sub_probs[0, sub_pred_idx].item()
            
            # Get all probabilities
            sub_probs_dict = {cls: prob.item() for cls, prob in zip(sub_classes, sub_probs[0])}
        
        return {
            'root_pred': root_pred,
            'root_conf': root_conf,
            'final_pred': sub_pred,
            'final_conf': sub_conf,
            'all_probs': sub_probs_dict
        }
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None

print("✓ Inference function siap digunakan")

## **5. Run Inference on External Data**

In [ ]:
if df_external is not None and len(df_external) > 0:
    print("="*70)
    print("RUNNING INFERENCE")
    print("="*70)
    
    predictions = []
    confidences = []
    
    print(f"\nProcessing {len(df_external)} images...")
    
    for idx, row in df_external.iterrows():
        if (idx + 1) % max(1, len(df_external) // 10) == 0:
            print(f"  {idx + 1}/{len(df_external)}")
        
        result = predict_hierarchical(
            row['image_path'],
            model_root,
            model_organic,
            model_inorganic,
            data_transforms
        )
        
        if result:
            predictions.append(result['final_pred'])
            confidences.append(result['final_conf'])
        else:
            predictions.append('ERROR')
            confidences.append(0.0)
    
    df_external['predicted_label'] = predictions
    df_external['confidence'] = confidences
    
    print("\n✓ Inference selesai!")
    
    # Show results
    print("\nSample hasil prediksi:")
    display_cols = ['image_path', 'predicted_label', 'confidence']
    if df_external['true_label'].notna().any():
        display_cols.insert(2, 'true_label')
    
    print(df_external[display_cols].head(10))
else:
    print("⚠️  Tidak ada data untuk di-inference")

## **6. Evaluation Metrics (Jika Ada Ground Truth)**

In [ ]:
if df_external is not None and df_external['true_label'].notna().any():
    print("="*70)
    print("EVALUATION METRICS")
    print("="*70)
    
    true_labels = df_external['true_label'].values
    pred_labels = df_external['predicted_label'].values
    confs = df_external['confidence'].values
    
    # Accuracy
    accuracy = accuracy_score(true_labels, pred_labels)
    print(f"\nOverall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    
    # Classification Report
    print("\n" + "="*70)
    print("CLASSIFICATION REPORT")
    print("="*70)
    print()
    report = classification_report(true_labels, pred_labels, digits=4)
    print(report)
    
    # Confusion Matrix
    cm = confusion_matrix(true_labels, pred_labels)
    
    # Get unique classes
    classes = sorted(np.unique(np.concatenate([true_labels, pred_labels])))
    
    print("\n" + "="*70)
    print("CONFUSION MATRIX")
    print("="*70)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_title(f'Confusion Matrix - External Test Data\n(Accuracy: {accuracy:.4f})')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n✓ Confusion matrix disimpan")
    
    # Confidence analysis
    print("\n" + "="*70)
    print("CONFIDENCE ANALYSIS")
    print("="*70)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Confidence distribution
    axes[0].hist(confs, bins=30, edgecolor='black', alpha=0.7, color='skyblue')
    axes[0].axvline(np.mean(confs), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(confs):.3f}')
    axes[0].set_title('Confidence Score Distribution')
    axes[0].set_xlabel('Confidence')
    axes[0].set_ylabel('Frequency')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Correct vs Incorrect predictions
    correct_mask = true_labels == pred_labels
    correct_confs = confs[correct_mask]
    incorrect_confs = confs[~correct_mask]
    
    axes[1].hist(correct_confs, bins=20, alpha=0.6, label=f'Correct (n={len(correct_confs)})', color='green')
    axes[1].hist(incorrect_confs, bins=20, alpha=0.6, label=f'Incorrect (n={len(incorrect_confs)})', color='red')
    axes[1].set_title('Confidence: Correct vs Incorrect Predictions')
    axes[1].set_xlabel('Confidence')
    axes[1].set_ylabel('Frequency')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'confidence_analysis.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nAverage Confidence:")
    print(f"  Overall: {np.mean(confs):.4f}")
    print(f"  Correct predictions: {np.mean(correct_confs):.4f}")
    print(f"  Incorrect predictions: {np.mean(incorrect_confs):.4f}")
    print(f"\n✓ Confidence analysis disimpan")
else:
    print("⚠️  Ground truth tidak tersedia, skip evaluation metrics")

## **7. Visualisasi Hasil Prediksi**

In [ ]:
if df_external is not None and len(df_external) > 0:
    print("="*70)
    print("VISUALISASI HASIL PREDIKSI")
    print("="*70)
    
    # Get sample images
    if df_external['true_label'].notna().any():
        correct_mask = df_external['predicted_label'] == df_external['true_label']
        
        # Correct predictions
        correct_imgs = df_external[correct_mask].sample(min(6, len(df_external[correct_mask])))
        
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        
        for idx, (_, row) in enumerate(correct_imgs.iterrows()):
            img = Image.open(row['image_path'])
            axes[idx].imshow(img)
            axes[idx].set_title(
                f"✓ CORRECT\n"
                f"Pred: {row['predicted_label']}\n"
                f"True: {row['true_label']}\n"
                f"Conf: {row['confidence']:.3f}",
                color='green', fontweight='bold'
            )
            axes[idx].axis('off')
        
        # Hide unused subplots
        for idx in range(len(correct_imgs), 6):
            axes[idx].axis('off')
        
        plt.suptitle('Sampel Prediksi Benar', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join(RESULTS_DIR, 'correct_predictions.png'), dpi=150, bbox_inches='tight')
        plt.show()
        
        # Incorrect predictions (jika ada)
        incorrect_mask = df_external['predicted_label'] != df_external['true_label']
        if incorrect_mask.any():
            incorrect_imgs = df_external[incorrect_mask].sample(min(6, len(df_external[incorrect_mask])))
            
            fig, axes = plt.subplots(2, 3, figsize=(15, 10))
            axes = axes.flatten()
            
            for idx, (_, row) in enumerate(incorrect_imgs.iterrows()):
                img = Image.open(row['image_path'])
                axes[idx].imshow(img)
                axes[idx].set_title(
                    f"✗ INCORRECT\n"
                    f"Pred: {row['predicted_label']}\n"
                    f"True: {row['true_label']}\n"
                    f"Conf: {row['confidence']:.3f}",
                    color='red', fontweight='bold'
                )
                axes[idx].axis('off')
            
            # Hide unused subplots
            for idx in range(len(incorrect_imgs), 6):
                axes[idx].axis('off')
            
            plt.suptitle('Sampel Prediksi Salah', fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.savefig(os.path.join(RESULTS_DIR, 'incorrect_predictions.png'), dpi=150, bbox_inches='tight')
            plt.show()
    else:
        # Tanpa ground truth, tampilkan random samples
        sample_imgs = df_external.sample(min(6, len(df_external)))
        
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        
        for idx, (_, row) in enumerate(sample_imgs.iterrows()):
            img = Image.open(row['image_path'])
            axes[idx].imshow(img)
            axes[idx].set_title(
                f"Pred: {row['predicted_label']}\n"
                f"Conf: {row['confidence']:.3f}",
                fontweight='bold'
            )
            axes[idx].axis('off')
        
        # Hide unused subplots
        for idx in range(len(sample_imgs), 6):
            axes[idx].axis('off')
        
        plt.suptitle('Sampel Hasil Prediksi', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join(RESULTS_DIR, 'sample_predictions.png'), dpi=150, bbox_inches='tight')
        plt.show()
    
    print("\n✓ Visualisasi prediksi disimpan")

## **8. Save Results to CSV**

In [ ]:
if df_external is not None:
    output_csv = os.path.join(RESULTS_DIR, 'external_test_results.csv')
    df_external.to_csv(output_csv, index=False)
    print(f"✓ Hasil disimpan ke {output_csv}")
    
    # Summary statistics
    summary = {
        'total_images': len(df_external),
        'avg_confidence': float(df_external['confidence'].mean()),
        'min_confidence': float(df_external['confidence'].min()),
        'max_confidence': float(df_external['confidence'].max()),
        'test_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    
    if df_external['true_label'].notna().any():
        correct = (df_external['predicted_label'] == df_external['true_label']).sum()
        summary['accuracy'] = float(correct / len(df_external))
        summary['correct_predictions'] = int(correct)
        summary['incorrect_predictions'] = int(len(df_external) - correct)
    
    # Save summary as JSON
    summary_json = os.path.join(RESULTS_DIR, 'test_summary.json')
    with open(summary_json, 'w') as f:
        json.dump(summary, f, indent=4)
    
    print(f"✓ Summary disimpan ke {summary_json}")
    
    print("\n" + "="*70)
    print("TEST SUMMARY")
    print("="*70)
    for key, value in summary.items():
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")

## **9. Insights & Rekomendasi**

In [ ]:
print("="*70)
print("INSIGHTS & REKOMENDASI")
print("="*70)

if df_external is not None and len(df_external) > 0:
    confs = df_external['confidence'].values
    
    print(f"""
╔══════════════════════════════════════════════════════════════════╗
║                        MODEL PERFORMANCE                          ║
╠══════════════════════════════════════════════════════════════════╣

📊 Confidence Statistics:
   • Average Confidence: {np.mean(confs):.4f}
   • Median Confidence:  {np.median(confs):.4f}
   • Std Deviation:      {np.std(confs):.4f}
   • Range:              [{np.min(confs):.4f}, {np.max(confs):.4f}]
""")
    
    # Confidence bins
    high_conf = (confs >= 0.9).sum()
    med_conf = ((confs >= 0.7) & (confs < 0.9)).sum()
    low_conf = (confs < 0.7).sum()
    
    print(f"""📈 Confidence Distribution:
   • High (≥0.9):  {high_conf:3d} ({100*high_conf/len(confs):.1f}%)
   • Medium (0.7-0.9): {med_conf:3d} ({100*med_conf/len(confs):.1f}%)
   • Low (<0.7):   {low_conf:3d} ({100*low_conf/len(confs):.1f}%)
""")
    
    if df_external['true_label'].notna().any():
        accuracy = (df_external['predicted_label'] == df_external['true_label']).mean()
        print(f"""✅ Accuracy: {accuracy:.4f} ({100*accuracy:.2f}%)\n""")
        
        correct_confs = df_external[df_external['predicted_label'] == df_external['true_label']]['confidence'].values
        incorrect_confs = df_external[df_external['predicted_label'] != df_external['true_label']]['confidence'].values
        
        if len(correct_confs) > 0 and len(incorrect_confs) > 0:
            print(f"""🎯 Prediction Confidence:
   • Correct predictions:   avg={np.mean(correct_confs):.4f}
   • Incorrect predictions: avg={np.mean(incorrect_confs):.4f}
   • Confidence gap: {np.mean(correct_confs) - np.mean(incorrect_confs):.4f}
""")
    
    print("""╠══════════════════════════════════════════════════════════════════╣
║                         REKOMENDASI                               ║
╚══════════════════════════════════════════════════════════════════╝

1. CONFIDENCE THRESHOLD:
   • Gunakan threshold ≥0.8 untuk high-confidence predictions
   • Require manual review untuk confidence 0.6-0.8
   • Reject/re-process untuk confidence <0.6

2. ERROR ANALYSIS:
   • Identifikasi misclassified samples dengan visualisasi
   • Analyze common confusion pairs
   • Collect hard examples untuk augmentation/retraining

3. MODEL IMPROVEMENT:
   • Apply data imbalance techniques (dari notebook 06)
   • Fine-tune dengan external data yang challenging
   • Consider ensemble dengan model architecture lain

4. DEPLOYMENT:
   • Monitor confidence distribution in production
   • Log predictions untuk periodic re-evaluation
   • Set up alert untuk sudden accuracy drops
""")
else:
    print("⚠️  Tidak ada data hasil untuk di-analyze")